# MuscleMap Thigh — MyoSegmenTUM Evaluation

Computes per-muscle metrics for MuscleMap **thigh** segmentations on:
1. **Water images** → `../segs_water/`
2. **Fat-fraction images** → `../segmentations_fat_frac/`

Ground truth: `../../myosegmenTUM/{subject}/SegmentationMasks/combined_gt_stack{N}.mha`

| Muscle | GT label | Thigh-model label (1–28 scheme) |
|---|---|---|
| L_gracilis | 1 | 11 |
| L_sartorius | 4 | 9 |
| R_gracilis | 5 | 12 |
| R_sartorius | 8 | 10 |

In [ ]:
import glob
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk

import sys
sys.path.insert(0, os.path.join('..', '..', '..', 'src'))
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [ ]:
BOUNDARY_DISTANCE = 1

WATER_SEG_DIR   = os.path.join('..', 'segs_water')
FATFRAC_SEG_DIR = os.path.join('..', 'segmentations_fat_frac')
GT_BASE         = os.path.join('..', '..', 'myosegmenTUM')

RESULT_WATER   = 'results_water'
RESULT_FATFRAC = 'results_fat_frac'
os.makedirs(RESULT_WATER,   exist_ok=True)
os.makedirs(RESULT_FATFRAC, exist_ok=True)

# (muscle_name, gt_label_idx, thigh_model_label_idx)
# GT labels from combined_gt_stack{N}.mha (MyoSegmenTUM scheme)
# Thigh model labels from 1-28 Zenodo scheme (same as asian notebook)
MUSCLES = [
    ('L_gracilis',  1, 11),
    ('L_sartorius', 4,  9),
    ('R_gracilis',  5, 12),
    ('R_sartorius', 8, 10),
]

print(f'Water segs  : {os.path.abspath(WATER_SEG_DIR)}')
print(f'FF segs     : {os.path.abspath(FATFRAC_SEG_DIR)}')
print(f'GT base     : {os.path.abspath(GT_BASE)}')

In [ ]:
def evaluate_muscle(muscle_name, gt_label_idx, mm_label_idx,
                    seg_files, modality_re, result_dir, csv_suffix):
    """
    modality_re: regex with two capture groups (subject, stack_num)
    e.g. r'(.+)_WATER_stack(\d+)_dseg\.nii\.gz'
    """
    results = []
    for seg_file in seg_files:
        m = re.match(modality_re, os.path.basename(seg_file))
        if not m:
            print(f'  could not parse filename: {seg_file}, skipping')
            continue
        subject   = m.group(1)
        stack_num = m.group(2)

        gt_path = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                               f'combined_gt_stack{stack_num}.mha')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label_idx, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        seg_sitk = sitk.ReadImage(seg_file)
        pred_arr = (sitk.GetArrayFromImage(seg_sitk) == mm_label_idx).astype(np.uint8)

        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {os.path.basename(seg_file)}: empty mask '
                  f'(gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'image':                                gt_path,
            'pred_label':                           os.path.basename(seg_file),
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df       = pd.DataFrame(results)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_{csv_suffix}.csv')
    df.to_csv(csv_path)
    print(f'  Saved {len(df)} rows -> {csv_path}')
    return df

## Water segmentations

In [ ]:
water_seg_files = sorted(glob.glob(os.path.join(WATER_SEG_DIR, '*_dseg.nii.gz')))
print(f'Water seg files: {len(water_seg_files)}')
for p in water_seg_files:
    print(' ', p)

In [ ]:
WATER_RE = r'(.+)_WATER_stack(\d+)_dseg\.nii\.gz'

dfs_water = {}
for muscle_name, gt_idx, mm_idx in MUSCLES:
    print(f'\n-- {muscle_name} (water) --')
    dfs_water[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, mm_idx,
        water_seg_files, WATER_RE,
        RESULT_WATER, 'musclemap_thigh_water',
    )

print('\nWater done.')

## Fat-fraction segmentations

In [ ]:
fatfrac_seg_files = sorted(glob.glob(os.path.join(FATFRAC_SEG_DIR, '*_dseg.nii.gz')))
print(f'Fat-fraction seg files: {len(fatfrac_seg_files)}')
for p in fatfrac_seg_files:
    print(' ', p)

In [ ]:
FATFRAC_RE = r'(.+)_FATFRACTION_stack(\d+)_dseg\.nii\.gz'

dfs_fatfrac = {}
for muscle_name, gt_idx, mm_idx in MUSCLES:
    print(f'\n-- {muscle_name} (fat fraction) --')
    dfs_fatfrac[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, mm_idx,
        fatfrac_seg_files, FATFRAC_RE,
        RESULT_FATFRAC, 'musclemap_thigh_fatfrac',
    )

print('\nFat-fraction done.')

## Results summary

In [ ]:
from IPython.display import display

for modality, dfs, result_dir, suffix in [
    ('water',        dfs_water,   RESULT_WATER,   'musclemap_thigh_water'),
    ('fat fraction', dfs_fatfrac, RESULT_FATFRAC, 'musclemap_thigh_fatfrac'),
]:
    print(f'\n=== {modality} ===')
    summary_rows = []
    for muscle_name, df in dfs.items():
        dice_col = f'{muscle_name}_dice'
        hd_col   = f'{muscle_name}_hausdorff'
        summary_rows.append({
            'muscle':         muscle_name,
            'n':              len(df),
            'dice_mean':      df[dice_col].mean(),
            'dice_std':       df[dice_col].std(),
            'hausdorff_mean': df[hd_col].mean(),
            'hausdorff_std':  df[hd_col].std(),
        })
    summary = pd.DataFrame(summary_rows).set_index('muscle')
    summary_path = os.path.join(result_dir, f'summary_{suffix}.csv')
    summary.to_csv(summary_path)
    print(f'Summary saved -> {summary_path}')
    display(summary.round(4))